In [23]:
!pip -q install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip -q install dgl==2.4.0+cu121 -f https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html
!pip -q install "numpy<2" --force-reinstall --no-cache-dir
!pip -q install "pydantic<2" "pytorch-ignite>=0.4.11,<0.5" "transformers==4.46.3" "jarvis-tools==2026.6.12" scikit-learn scipy matplotlib huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 40.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 138.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 194.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-pyth

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, dgl, pydantic, transformers, ignite, numpy
print("torch", torch.__version__)
print("dgl", dgl.__version__)
print("numpy", numpy.__version__)                  # must start with 1.
print("pydantic", pydantic.VERSION)                # must start with 1.
print("transformers", transformers.__version__)    # expect 4.46.3
print("ignite", ignite.__version__)
assert torch.cuda.is_available(), "No GPU"
_g = dgl.graph(([0,1,2],[1,2,0])).to("cuda")
print("DGL CUDA OK:", _g.device)
from jarvis.core.atoms import get_supercell_dims
from transformers import AutoTokenizer, AutoModel
print("ALL CLEAR")

/usr/local/lib/python3.12/dist-packages/ignite/handlers/checkpoint.py:17: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


torch 2.4.0+cu121
dgl 2.4.0+cu121
numpy 1.26.4
pydantic 1.10.26
transformers 4.46.3
ignite 0.4.13
DGL CUDA OK: cuda:0
ALL CLEAR


In [3]:
import sys, subprocess, pathlib

WORK = pathlib.Path("/content")
REPO = WORK / "crysmmnet"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/kdmsit/crysmmnet.git", str(REPO)], check=True)

SRC = REPO / "src"

stale = SRC / "profile.py"
if stale.exists():
    stale.rename(SRC / "_profile_unused.py.bak")

os.chdir(SRC)

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("cwd:", os.getcwd())
assert (SRC / "vocab_mappings.txt").exists()

cwd: /content/crysmmnet/src


In [4]:
from graphs import StructureDataset, prepare_line_graph_batch
from data import get_id_train_val_test
from config import TrainingConfig
from train import train_dgl

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/323 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at m3rg-iitd/matscibert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
import os, sys, json
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader

In [6]:
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id="Godseye1311/alignn-band-gap",
    repo_type="dataset",
    allow_patterns=["materials_tabular.csv", "text_data.csv", "alignn_graphs/*"],
)
print("Downloaded to:", local_dir)

Fetching 156 files:   0%|          | 0/156 [00:00<?, ?it/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--Godseye1311--alignn-band-gap/snapshots/0522258af445efa653a16e56a2d109c2a59b62fe


In [7]:
graphs_dir = os.path.join(local_dir, "alignn_graphs")

with open(os.path.join(graphs_dir, "graph_shards.json")) as f:
    shard_index = json.load(f)

print("Materials with a graph:", len(shard_index))
shards_used = sorted(set(v["shard"] for v in shard_index.values()))
print("Shards required:", len(shards_used))
print(shards_used[:5], "...")

Materials with a graph: 101161
Shards required: 51
['shard_0000000', 'shard_0002000', 'shard_0004000', 'shard_0006000', 'shard_0008000'] ...


In [8]:
tabular = pd.read_csv(os.path.join(local_dir, "materials_tabular.csv"))
text_df = pd.read_csv(os.path.join(local_dir, "text_data.csv"))

print("materials_tabular.csv columns:", list(tabular.columns))
print("text_data.csv columns:", list(text_df.columns))

df = tabular.merge(text_df, on="material_id", how="inner")

# Keep only rows that actually have a graph, and a non-null band_gap target
df = df[df["material_id"].isin(shard_index.keys())]
df = df[df["band_gap"].notna()]

df = df.rename(columns={
    "material_id": "jid",
    "band_gap": "target",
    "description": "text",
})
df = df[["jid", "target", "text"]].reset_index(drop=True)

print("\nUsable rows (have graph + text + band_gap):", len(df))
print(df.head())

materials_tabular.csv columns: ['material_id', 'formula_pretty', 'density', 'density_atomic', 'total_magnetization', 'num_magnetic_sites', 'chemsys', 'ordering', 'nelements', 'nsites', 'formation_energy_per_atom', 'band_gap', 'symmetry', 'space_group_number', 'lattice_a', 'lattice_b', 'lattice_c', 'lattice_alpha', 'lattice_beta', 'lattice_gamma', 'point_group', 'point_group_order', 'is_centrosymmetric', 'bravais_lattice', 'k_vrh', 'g_vrh', 'universal_anisotropy']
text_data.csv columns: ['material_id', 'description']

Usable rows (have graph + text + band_gap): 99517
          jid  target                                               text
0    mp-10018     0.0  Ac is Copper structured and crystallizes in th...
1   mp-862690     0.0  Ac is alpha La structured and crystallizes in ...
2  mp-1183057     0.0  Ac is Copper-like structured and crystallizes ...
3  mp-1183069     0.0  Ac is alpha Samarium structured and crystalliz...
4   mp-861724     0.0  Ac₂IrAg is Heusler structured and cryst

In [9]:
class ShardedGraphTextDataset(Dataset):
    def __init__(self, df, shard_index, graphs_dir):
        self.df = df.reset_index(drop=True)
        self.shard_index = shard_index
        self.graphs_dir = graphs_dir
        self._shard_cache = {}
        self.labels = torch.tensor(self.df["target"].values).type(torch.get_default_dtype())
        self.ids = self.df["jid"]
        self.text = self.df["text"]
        self.prepare_batch = None

    def _load_shard(self, shard_name):
        if shard_name not in self._shard_cache:
            atoms, _ = dgl.load_graphs(os.path.join(self.graphs_dir, f"{shard_name}_atom.bin"))
            lines, _ = dgl.load_graphs(os.path.join(self.graphs_dir, f"{shard_name}_line.bin"))
            self._shard_cache[shard_name] = (atoms, lines)
        return self._shard_cache[shard_name]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        loc = self.shard_index[row["jid"]]
        atoms, lines = self._load_shard(loc["shard"])
        g = atoms[loc["index"]]
        lg = lines[loc["index"]]
        label = self.labels[idx]
        text = self.text[idx]
        return g, lg, text, label

    @staticmethod
    def collate_line_graph(samples):
        return StructureDataset.collate_line_graph(samples)  # unmodified, from graphs.py

In [10]:
with open("config.json") as f:
    cfg_dict = json.load(f)

cfg_dict["dataset"] = "user_data"
cfg_dict["target"] = "target"
cfg_dict["output_dir"] = "./outputs/"
cfg_dict["epochs"] = 1000
cfg_dict["batch_size"] = 64

config = TrainingConfig(**cfg_dict)

id_train, id_val, id_test = get_id_train_val_test(
    total_size=len(df),
    split_seed=config.random_seed,
    train_ratio=0.8, val_ratio=0.1, test_ratio=0.1,
    keep_data_order=config.keep_data_order,
)

train_ds = ShardedGraphTextDataset(df.iloc[id_train], shard_index, graphs_dir)
val_ds   = ShardedGraphTextDataset(df.iloc[id_val],   shard_index, graphs_dir)
test_ds  = ShardedGraphTextDataset(df.iloc[id_test],  shard_index, graphs_dir)

from graphs import prepare_line_graph_batch  # unmodified, from graphs.py
for ds in (train_ds, val_ds, test_ds):
    ds.prepare_batch = prepare_line_graph_batch

train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                           collate_fn=ShardedGraphTextDataset.collate_line_graph,
                           drop_last=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False,
                         collate_fn=ShardedGraphTextDataset.collate_line_graph,
                         drop_last=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False,
                          collate_fn=ShardedGraphTextDataset.collate_line_graph,
                          drop_last=False, num_workers=0)

print("train/val/test:", len(train_ds), len(val_ds), len(test_ds))

train/val/test: 79613 9951 9951


In [11]:
import torch
import train
import models.alignn as alignn_module

train.device = torch.device("cpu")
alignn_module.device = torch.device("cpu")
alignn_module.text_model.to("cpu")
device = torch.device("cpu")

print("models.alignn device:", alignn_module.device)
print("text_model device:", next(alignn_module.text_model.parameters()).device)
print("train.py device:", train.device)

models.alignn device: cpu
text_model device: cpu
train.py device: cpu


In [18]:
import torch, models.alignn as alignn_mod
alignn_mod.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alignn_mod.text_model.to(alignn_mod.device)
print("models.alignn device:", alignn_mod.device,
      "| text_model actually on:", next(alignn_mod.text_model.parameters()).device)

models.alignn device: cuda | text_model actually on: cuda:0


In [ ]:
history = train_dgl(
    config,
    train_val_test_loaders=[train_loader, val_loader, test_loader, prepare_line_graph_batch],
    resume=0,
)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:117: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:185.)
  return F.linear(input, self.weight, self.bias)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:768: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBL

[1/1243]   0%|           [00:00<?]